[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs/Lab_6_Mars_Rover_Capstone.ipynb)


In [ ]:
# 🚀 GOOGLE COLAB SETUP
# If you are running this notebook on Google Colab, execute this cell to clone the repo and set paths:
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        os.chdir('PDE-Rover-Engine')
        if '.' not in sys.path:
            sys.path.insert(0, '.')
        print('[OK] Google Colab setup complete! Connected to PDE-Rover-Engine repository.')
except NameError:
    pass


# 🚀 Lab 6: The Capstone: Discretization & Autonomous Mars Navigation
### *Mission Control: The Flight Computer Upload*
**Course:** Applied Partial Differential Equations & Planetary Robotics  
**Topic:** The Grand Scale-Up (Connecting Syllabus to Planetary Robotics)

---
### 🧭 The Rosetta Stone: Translating Math into English and Code
| 1. Professor's Blackboard Math | 2. Plain Human English | 3. Python Code |
| :--- | :--- | :--- |
| $\nabla^2\phi = 0$ | "Continuous Laplace's equation from Lab 4." | `(D_x^2 + D_y^2)phi = 0` |
| $\phi_{i,j} = \frac{1}{4}(\phi_{\text{up}} + \phi_{\text{down}} + \dots)$ | "5-point discrete Mean Value stencil." | `avg = 0.25 * (roll_up + roll_down + ...)` |
| $\frac{\partial\phi}{\partial n} = 0$ | "Zero-flux Neumann walls: water cannot cross craters." | `phi[~free] = 0; cnt = count_active_neighbors` |
| $\omega_{\text{opt}} = \frac{2}{1 + \sin(\pi/N)}$ | "Optimal SOR over-relaxation parameter." | `phi = (1 - w)*old + w*avg` |

---
## 1. In Plain English: "From Paper Formulas to NASA Elevation Maps"
In Labs 1–5, you solved PDEs on paper.  
**The Crisis:** When our rover lands in Kasei Valles on Mars, the terrain has **10,000 irregular crater walls**! There is no simple paper formula for $\phi_1$ and $\phi_2$ that can satisfy 10,000 irregular rocks.  
**The Solution:** We convert $(D_x^2 + D_y^2)\phi = 0$ into a **discrete 5-point stencil** and let Python compute the values across 25,600 nodes!

---
## 2. The "WHY": Why do we do this method? What does it help with?
* **Why do we do it?** Paper formulas fail on complex geometry.
* **What it helps with:** It allows the rover's computer to calculate the harmonic navigation field in **2.5 seconds** across real Mars elevation models.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab6

%matplotlib inline
print("Lab 6 Capstone Initialized! Mission Launch Sequence Ready.")


---
## 3. The Mathematical Recipe: The 5-Point Stencil

### Taylor Series Discretization:
$$\frac{\partial^2\phi}{\partial x^2} \approx \frac{\phi_{i+1, j} - 2\phi_{i,j} + \phi_{i-1, j}}{h^2}, \quad \frac{\partial^2\phi}{\partial y^2} \approx \frac{\phi_{i, j+1} - 2\phi_{i,j} + \phi_{i, j-1}}{h^2}$$
Adding them together and multiplying by $h^2$:
$$\phi_{i,j} = \frac{1}{4}\left(\phi_{i+1, j} + \phi_{i-1, j} + \phi_{i, j+1} + \phi_{i, j-1}\right)$$
> *"Every cell is simply the arithmetic average of its 4 orthogonal neighbors!"*


---
## 4. 📝 Paper & Pen Challenge: The 4-Node Hand Calculation

> 💡 **Instructions:** Grab your pen and paper!  
> *(If you get stuck, open `INSTRUCTOR_ANSWER_KEY.md` Section 6.1!)*

Consider a small $4 \times 4$ grid:
* Top boundary (Start / Source): $\phi = 1.0$
* Bottom boundary (Goal / Sink): $\phi = 0.0$
* Left & Right boundaries: $\phi = 0.5$
* Interior cells $A, B, C, D$ initialized to $0.0$.

Compute Sweep 1 for **Cell A** ($\phi_{1,1}$) and **Cell B** ($\phi_{1,2}$):
$$\phi_A^{(1)} = \frac{1.0 + 0.0 + 0.5 + 0.0}{4} = ?$$

Enter your hand-calculated decimals below:


In [ ]:
# GRADED CELL: 4-Node Hand Calculation
### START CODE HERE ###
phi_A = 0.375
phi_B = 0.375
### END CODE HERE ###

# Run verification test:
test_lab6.check_paper_challenge_1(phi_A, phi_B)


---
## 5. 🚀 The Rover Bridge Challenge: The 25,600-Node Scale-Up

On paper, calculating 4 cells took 2 minutes. On an $N = 16$ grid, what is the theoretical optimal relaxation factor $\omega_{\text{opt}}$ given by Young's formula?
$$\omega_{\text{opt}} = \frac{2}{1 + \sin\left(\frac{\pi}{N}\right)}$$

Compute $\omega_{\text{opt}}$ for $N = 16$ on your paper and enter it below:


In [ ]:
# GRADED CELL: Optimal Omega for N=16
### START CODE HERE ###
omega_16 = 2.0 / (1.0 + np.sin(np.pi / 16))
### END CODE HERE ###

# Run verification test:
test_lab6.check_paper_challenge_2(omega_16)


---
## 6. 💻 Python Implementation: SOR with Zero-Flux Neumann Walls

Let's implement the full Red-Black Successive Over-Relaxation (SOR) solver with zero-flux Neumann walls ($\frac{\partial\phi}{\partial n} = 0$) on crater obstacles!


In [ ]:
# GRADED FUNCTION: sor_neumann_step
def sor_neumann_step(phi, fixed, free, omega=1.5, tol=1e-5, maxit=5000):
    # Red-Black SOR solver with zero-flux Neumann crater walls
    phi = phi.copy()
    n, m = phi.shape
    f = free.astype(float)
    
    # Active neighbor count
    cnt = sum(np.roll(f, s, a) for a in (0, 1) for s in (1, -1))
    cnt = np.maximum(cnt, 1.0)
    
    yy, xx = np.mgrid[0:n, 0:m]
    parity = (xx + yy) % 2
    masks = [(parity == c) & free & ~fixed for c in (0, 1)]
    phi[~free] = 0.0
    hist = []
    
    ### START CODE HERE ###
    for _ in range(maxit):
        d = 0.0
        for mk in masks:
            pf = phi * f
            avg = (
                np.roll(pf, 1, 0) + np.roll(pf, -1, 0)
                + np.roll(pf, 1, 1) + np.roll(pf, -1, 1)
            ) / cnt
            old = phi[mk]
            phi[mk] = (1 - omega) * old + omega * avg[mk]
            d = max(d, float(np.abs(phi[mk] - old).max()))
        hist.append(d)
        if d < tol:
            break
    ### END CODE HERE ###
    
    return phi, hist

# Run automated unit test:
test_lab6.test_sor_neumann_step(sor_neumann_step)


---
## 7. 🌉 The Mathematical & Physical Bridge: Why Do We Replace Classical Equations with a PDE?

Before running the simulation on **25,600 Martian terrain nodes**, let us understand the deep link connecting the blackboard mathematics from your semester syllabus directly to the physical robotics of space exploration.

### 1. What equation does classical robotics start with? (Artificial Potential Fields - APF)
In standard robotics and video game navigation, engineers start with an intuitive algebraic formula called **Artificial Potential Fields (APF)**:
$$U(\mathbf{x}) = U_{\text{att}}(\mathbf{x}) + U_{\text{rep}}(\mathbf{x})$$

* **Attractive Well ($U_{\text{att}}$):** The target science goal pulls the rover like a gravitational spring:
  $$U_{\text{att}}(\mathbf{x}) = \frac{1}{2} k_{\text{att}} \|\mathbf{x} - \mathbf{x}_{\text{goal}}\|^2 \implies \mathbf{F}_{\text{att}} = -\nabla U_{\text{att}} = -k_{\text{att}}(\mathbf{x} - \mathbf{x}_{\text{goal}})$$
* **Repulsive Wall ($U_{\text{rep}}$):** Every crater cliff pushes the rover away like an electrostatic charge:
  $$U_{\text{rep}}(\mathbf{x}) = \frac{1}{2} k_{\text{rep}} \left(\frac{1}{d(\mathbf{x})} - \frac{1}{d_0}\right)^2 \implies \mathbf{F}_{\text{rep}} = -\nabla U_{\text{rep}}$$

The rover then acts like a rolling marble following the net force $\mathbf{F}_{\text{net}} = -\nabla U = \mathbf{F}_{\text{att}} + \mathbf{F}_{\text{rep}}$.

---

### 2. Why does this equation FAIL physically? (The Fatal Local Minimum Trap 🪤)
Imagine the Ares Rover enters a U-shaped crater basin or a narrow dead-end canyon on Mars, with the science exploration target located directly on the other side of the canyon wall:
* As the rover drives into the canyon toward the target, the forward pull toward the goal ($\mathbf{F}_{\text{att}}$) increases.
* Simultaneously, the backward repulsion pushing away from the crater wall ($\mathbf{F}_{\text{rep}}$) increases.
* Inside the canyon, there is a point where the two forces are **equal in magnitude and opposite in direction**:
  $$\mathbf{F}_{\text{att}} + \mathbf{F}_{\text{rep}} = \mathbf{0} \implies \nabla U = \mathbf{0}$$

> ⚠️ **The Catastrophe:** Even though the rover is miles away from the goal, the forces cancel out completely! The potential field forms an **interior pothole (local minimum)**. The rover stalls, oscillates, or freezes forever in the crater until its batteries drain.

---

### 3. Why do we need to CHANGE it using a Partial Differential Equation?
Why did APF fail? Because simple algebraic sums $U_{\text{att}} + U_{\text{rep}}$ allow the field to bend into arbitrary bowls and saddle points ($\nabla^2 U > 0$).

To guarantee that a rover will **NEVER get trapped**, we need a mathematical field that is *physically incapable of forming potholes in empty space*.

**Look to Nature for the Answer: The Physics of Water 🌊**  
If you pour water into a complex maze of rocks with a garden hose at the start and an open drain at the end:
* Does water ever get stuck in a dead-end pocket? **No!**
* Water fills the dead-end, pressure builds up against the rock walls, and the current naturally bends around the obstacle and flows cleanly into the drain.
* Why? Because in a steady-state fluid, **pressure diffuses smoothly everywhere**. There are no stagnant low-pressure "potholes" in the middle of a flowing river.

---

### 4. What is the PDE equation that replaces it, and what is its role?
The fundamental equation governing steady-state fluid flow, heat distribution, and electrostatic potential is **Laplace's Equation**:
$$\nabla^2 \phi = \frac{\partial^2 \phi}{\partial x^2} + \frac{\partial^2 \phi}{\partial y^2} = 0$$

#### Its Role in the Rover Project:
1. **The Domain ($\Omega$):** We map the Martian surface into a 2D scalar field $\phi(x, y)$.
2. **The Source & Sink (Dirichlet Boundary Conditions):**
   * **Start (Landing Site):** Fixed at highest potential $\phi = 1.0$ (like a high-pressure water pump).
   * **Goal (Science Site):** Fixed at lowest potential $\phi = 0.0$ (like an open suction drain).
3. **The Crater Obstacles (Neumann Boundary Conditions):**
   * Crater walls are declared **insulated boundaries** where no fluid can enter the rock:
     $$\frac{\partial \phi}{\partial n} = \nabla\phi \cdot \hat{\mathbf{n}} = 0$$
4. **The Mathematical Guarantee (The Strong Maximum Principle from Chapter 1.4):**
   As proved in your lecture notes (Page 33), a harmonic function $\phi$ satisfying $\nabla^2\phi = 0$ **cannot have local maxima or local minima anywhere in the interior of the domain**.
   * Extrema exist **strictly on the boundaries** ($\phi=1$ at the start, $\phi=0$ at the goal).
   * Therefore, the gradient $-\nabla\phi$ **NEVER vanishes** in free space!
5. **The Trajectory Curve (Lagrange's Streamline Method from Chapter 1.3):**
   The rover simply rides the current vector field $\mathbf{v} = -\nabla\phi$. By Lagrange's characteristic equation:
   $$\frac{dx}{u} = \frac{dy}{v}$$
   the rover traces the exact physical fluid streamline directly to the goal with a **100% mathematical guarantee of never getting stuck**!

---

### 🧭 Summary: Classical APF vs. Laplace PDE

| Feature | Classical Robotics (APF) | Laplace PDE Formulation (Our Project) |
| :--- | :--- | :--- |
| **Governing Equation** | $U = U_{\text{att}} + U_{\text{rep}}$ (Heuristic algebraic sum) | $\nabla^2\phi = 0$ (Elliptic 2nd-order PDE) |
| **Physical Model** | Rolling marble on a bumpy rubber sheet | Steady-state fluid flow / heat conduction |
| **Interior Potholes (Traps)** | **Common failure** ($\nabla U = \mathbf{0}$ inside craters) | **Impossible** (Guaranteed by the Maximum Principle) |
| **Obstacle Boundaries** | Repulsive push causes jitter and chattering | Neumann no-flux wall ($\partial\phi/\partial n = 0$): streamlines glide smoothly around walls |
| **Autonomous Trajectory** | Heuristic descent (often stuck) | Characteristic streamlines ($\frac{dx}{u} = \frac{dy}{v}$ via Lagrange) |

---
## 8. 🎨 Grand Mission Finale: Autonomous Navigation on Planetary Terrain

Now that we understand the physics and the mathematics, let us execute the simulation on **25,600 Martian elevation nodes**!


In [ ]:
from rover_path_finder.terrain import make_terrain
from rover_path_finder.planners import neumann_potential, descend, apf_potential, trap_map

# 1. Generate 100x100 Martian terrain
terrain_data = make_terrain(N=100, seed=3, rover_radius=2.0)
terrain = terrain_data['h']
hazards = terrain_data['hazard']
start = terrain_data['start']
goal = terrain_data['goal']

# 2. Solve Laplace's Equation
phi, sweeps, elapsed = neumann_potential(hazards, start=start, goal=goal)

# 3. Follow steepest gradient descent
path, reached = descend(phi, hazards, start=start, goal=goal)

# 4. Plot Navigation Field & Path
fig, ax = plt.subplots(figsize=(10, 8))
cs = ax.contourf(phi, levels=30, cmap='viridis')
plt.colorbar(cs, label="Harmonic Potential $\\phi(x, y)$")
ax.contour(hazards, levels=[0.5], colors='red', linewidths=1.5)

px = [p[1] for p in path]
py = [p[0] for p in path]
ax.plot(px, py, color='cyan', linewidth=3, label="Rover Trajectory")
ax.plot(start[1], start[0], 'go', markersize=10, label="Start ($\phi=1$)")
ax.plot(goal[1], goal[0], 'r*', markersize=14, label="Goal ($\phi=0$)")

ax.set_title("Ares Rover Navigation: Laplace Streamlines Around Martian Craters", fontsize=14)
ax.legend(loc='upper left')
plt.show()

print(f"Mission Status: Reached Goal? {reached} in {len(path)} steps ({elapsed:.2f}s)!")


## 9. 🪤 The Ultimate Benchmark: The U-Shaped Trap Test

Let's test our Laplace solver against classic Artificial Potential Fields (APF) inside a dead-end canyon:


In [ ]:
u_trap, trap_start, trap_goal = trap_map(N=100)

# APF
apf_field = apf_potential(u_trap, goal=trap_goal)
apf_path, apf_reached = descend(apf_field, u_trap, start=trap_start, goal=trap_goal, maxsteps=200)

# Laplace
laplace_field, _, _ = neumann_potential(u_trap, start=trap_start, goal=trap_goal)
laplace_path, laplace_reached = descend(laplace_field, u_trap, start=trap_start, goal=trap_goal, maxsteps=400)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# APF Plot
axes[0].contourf(apf_field, levels=25, cmap='magma')
axes[0].plot([p[1] for p in apf_path], [p[0] for p in apf_path], 'r-', linewidth=3, label="APF Path")
axes[0].plot(trap_start[1], trap_start[0], 'go', markersize=8)
axes[0].plot(trap_goal[1], trap_goal[0], 'y*', markersize=12)
axes[0].set_title("Classical Robotics (APF): TRAPPED in Local Minimum! ❌", fontsize=12)
axes[0].legend()

# Laplace Plot
axes[1].contourf(laplace_field, levels=25, cmap='viridis')
axes[1].plot([p[1] for p in laplace_path], [p[0] for p in laplace_path], 'c-', linewidth=3, label="Laplace Path")
axes[1].plot(trap_start[1], trap_start[0], 'go', markersize=8, label="Start")
axes[1].plot(trap_goal[1], trap_goal[0], 'y*', markersize=12, label="Goal")
axes[1].set_title("Laplace PDE: Maneuvers Cleanly Out of the Trap! ✅", fontsize=12)
axes[1].legend()

plt.tight_layout()
plt.show()

print("Benchmark Results:")
print(f"  • Classical APF: Reached target? {apf_reached} (Permanently trapped inside crater)")
print(f"  • Laplace PDE  : Reached target? {laplace_reached} (Safely escaped the trap in {len(laplace_path)} steps!)")


## 🏆 Course Graduation

**Congratulations on completing the Applied PDE Course!**

You have mastered the complete journey:
1. **Analytical Foundations:** Derived PDEs by eliminating constants, solved quasi-linear equations via Lagrange streamlines, conquered non-linear wavefronts with Charpit's method, and mastered differential operator calculus with Laplace's equation.
2. **Computational Scale-Up:** Discretized continuous equations into the 5-point stencil, proved convergence on paper, and accelerated relaxation by 66× using Young's SOR theorem.
3. **Planetary Robotics:** Solved the local minimum trap on Mars, ensuring autonomous space rovers can navigate any terrain safely!
